In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ga-customer-revenue-prediction/sample_submission.csv
/kaggle/input/ga-customer-revenue-prediction/train_v2.csv
/kaggle/input/ga-customer-revenue-prediction/test_v2.csv
/kaggle/input/ga-customer-revenue-prediction/sample_submission_v2.csv
/kaggle/input/ga-customer-revenue-prediction/train.csv
/kaggle/input/ga-customer-revenue-prediction/test.csv


In [2]:
import os, json, shutil
import pandas as pd
from pandas import json_normalize

# 1) 设置数据路径：把下面路径改成你 Notebook 里实际看到的路径
DATA_PATH = "/kaggle/input/ga-customer-revenue-prediction/train_v2.csv"

OUT_DIR = "/kaggle/working/evidence_pack"
os.makedirs(OUT_DIR, exist_ok=True)

JSON_COLS = ["device", "geoNetwork", "totals", "trafficSource"]

def safe_json_load(x):
    if pd.isna(x) or x == "":
        return {}
    try:
        return json.loads(x)
    except Exception:
        return {}

# 2) 读取：只解析你需要的列（避免内存爆）
USE_COLS = ["date", "fullVisitorId", "channelGrouping"] + JSON_COLS

df = pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype={"fullVisitorId": "string"},
    converters={c: safe_json_load for c in JSON_COLS},
)

# 3) flatten JSON 列（只展开一层即可满足 E1–E6）
for c in JSON_COLS:
    flat = json_normalize(df[c])
    flat.columns = [f"{c}.{cc}" for cc in flat.columns]
    df = df.drop(columns=[c]).join(flat)

# 4) 清洗关键指标
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")

# revenue 通常是微单位（字符串），先转数值；你可以保持原单位或除以 1e6
df["totals.transactionRevenue"] = pd.to_numeric(df.get("totals.transactionRevenue"), errors="coerce").fillna(0.0)
df["totals.transactions"] = pd.to_numeric(df.get("totals.transactions"), errors="coerce").fillna(0.0)

df["has_txn"] = (df["totals.transactions"] > 0).astype(int)
df["revenue"] = df["totals.transactionRevenue"]  # 如需换成常见单位：df["revenue"] = df["totals.transactionRevenue"] / 1e6

def add_evidence_id(dfx, prefix):
    dfx = dfx.reset_index(drop=True)
    dfx.insert(0, "evidence_id", [f"{prefix}_{i+1:02d}" for i in range(len(dfx))])
    return dfx

# ---------- E1 Overall ----------
E1 = pd.DataFrame([{
    "sessions": len(df),
    "transactions_sessions": int(df["has_txn"].sum()),
    "revenue": float(df["revenue"].sum()),
}])
E1["CVR"] = E1["transactions_sessions"] / E1["sessions"]
E1 = add_evidence_id(E1, "E1_row")
E1.to_csv(f"{OUT_DIR}/E1_overall.csv", index=False)

# ---------- E2 Trend by week ----------
tmp = df.dropna(subset=["date"]).copy()
tmp["week"] = tmp["date"].dt.to_period("W").astype(str)
E2 = tmp.groupby("week", as_index=False).agg(
    sessions=("fullVisitorId", "size"),
    transactions_sessions=("has_txn", "sum"),
    revenue=("revenue", "sum"),
)
E2["CVR"] = E2["transactions_sessions"] / E2["sessions"]
E2 = E2.sort_values("week")
E2 = add_evidence_id(E2, "E2_wk")
E2.to_csv(f"{OUT_DIR}/E2_trend_by_week.csv", index=False)

# ---------- E3 Channel ----------
E3 = df.groupby("channelGrouping", as_index=False).agg(
    sessions=("fullVisitorId", "size"),
    transactions_sessions=("has_txn", "sum"),
    revenue=("revenue", "sum"),
)
E3["CVR"] = E3["transactions_sessions"] / E3["sessions"]
E3["revenue_per_session"] = E3["revenue"] / E3["sessions"]
E3 = E3.sort_values("revenue", ascending=False)
E3 = add_evidence_id(E3, "E3_ch")
E3.to_csv(f"{OUT_DIR}/E3_channel.csv", index=False)

# ---------- E4 Source / Medium (Top 20) ----------
src = df.get("trafficSource.source", pd.Series(["(missing)"] * len(df)))
med = df.get("trafficSource.medium", pd.Series(["(missing)"] * len(df)))
df["source"] = src.fillna("(missing)")
df["medium"] = med.fillna("(missing)")

E4 = df.groupby(["source", "medium"], as_index=False).agg(
    sessions=("fullVisitorId", "size"),
    transactions_sessions=("has_txn", "sum"),
    revenue=("revenue", "sum"),
)
E4["CVR"] = E4["transactions_sessions"] / E4["sessions"]
E4 = E4.sort_values("sessions", ascending=False).head(20)
E4 = add_evidence_id(E4, "E4_sm")
E4.to_csv(f"{OUT_DIR}/E4_source_medium_top20.csv", index=False)

# ---------- E5 Device ----------
dev = df.get("device.deviceCategory", pd.Series(["(missing)"] * len(df))).fillna("(missing)")
df["deviceCategory"] = dev

E5 = df.groupby("deviceCategory", as_index=False).agg(
    sessions=("fullVisitorId", "size"),
    transactions_sessions=("has_txn", "sum"),
    revenue=("revenue", "sum"),
)
E5["CVR"] = E5["transactions_sessions"] / E5["sessions"]
E5 = E5.sort_values("sessions", ascending=False)
E5 = add_evidence_id(E5, "E5_dev")
E5.to_csv(f"{OUT_DIR}/E5_device.csv", index=False)

# ---------- E6 Country (Top 10) ----------
cty = df.get("geoNetwork.country", pd.Series(["(missing)"] * len(df))).fillna("(missing)")
df["country"] = cty

E6 = df.groupby("country", as_index=False).agg(
    sessions=("fullVisitorId", "size"),
    transactions_sessions=("has_txn", "sum"),
    revenue=("revenue", "sum"),
)
E6["CVR"] = E6["transactions_sessions"] / E6["sessions"]
E6 = E6.sort_values("sessions", ascending=False).head(10)
E6 = add_evidence_id(E6, "E6_cty")
E6.to_csv(f"{OUT_DIR}/E6_country_top10.csv", index=False)

# 5) 生成一份 summary（方便复制给 ChatGPT）
def md_table(path, n=10):
    t = pd.read_csv(path).head(n)
    # Kaggle 通常有 tabulate；如果没有，会报错，那就改用 to_string
    try:
        return t.to_markdown(index=False)
    except Exception:
        return t.to_string(index=False)

summary_path = "/kaggle/working/evidence_summary.md"
with open(summary_path, "w", encoding="utf-8") as f:
    for name in [
        "E1_overall.csv",
        "E2_trend_by_week.csv",
        "E3_channel.csv",
        "E4_source_medium_top20.csv",
        "E5_device.csv",
        "E6_country_top10.csv",
    ]:
        f.write(f"\n## {name}\n\n")
        f.write(md_table(f"{OUT_DIR}/{name}", n=10))
        f.write("\n\n")

# 6) 打包 zip：一次性下载
zip_path = "/kaggle/working/evidence_pack.zip"
shutil.make_archive("/kaggle/working/evidence_pack", "zip", OUT_DIR)

print("Done. Outputs:")
print("-", OUT_DIR)
print("-", summary_path)
print("-", zip_path)

Done. Outputs:
- /kaggle/working/evidence_pack
- /kaggle/working/evidence_summary.md
- /kaggle/working/evidence_pack.zip
